# LRRM

本示例演示如何使用 `skrf` 的 LRRM 校准。LRRM 代表线性-反射-反射-匹配，这些是校准所需的校准标准。存在几种不同的 LRRM 实现，它们使用略微不同的假设和匹配模型。`skrf` 的 LRRM 校准使用以下假设：* 线性标准的阻抗需要精确已知。* 第一个反射的相位需要在 90 度以内。它可以是有损的，并且假设在两个端口上是相同的。* 第二个反射的 |S11| 假设是已知的，其相位也需要在 90 度以内，并且假设在两个端口上是相同的。这两个反射需要不同，它们的相位差应为 180 度，以获得最佳精度。* 匹配被假设为串联的已知电阻和未知电感。匹配只需要在第一个端口上进行测量。校准标准和测量必须按照上述顺序提供给校准程序。如果遵循上述假设，则校准可以精确地求解反射、匹配和校准参数。

## 使用合成数据的 LRRM 示例

### 设置

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import skrf
from skrf.calibration import LRRM, TwelveTerm, terminate
from skrf.media import Coaxial
from skrf.network import two_port_reflect

skrf.stylely()

### 生成示例数据

我们首先生成一些合成误差模型和校准标准。我们将有两组校准标准：一组是用于校准的实际标准，它包含寄生参数；另一组是不包含寄生参数的近似标准，我们将提供给校准算法。

In [ ]:
freq = skrf.Frequency(1, 100, 100, 'GHz')

# 1.0 mm coaxial media for calibration error boxes
coax = Coaxial(freq, z0_port=50, Dint=0.44e-3, Dout=1.0e-3, sigma=1e8)

# Generate random error boxes
X = coax.random(n_ports=2, name='X')
Y = coax.random(n_ports=2, name='Y')

# Random switch terms
gamma_f = coax.random(n_ports=1, name='gamma_f')
gamma_r = coax.random(n_ports=1, name='gamma_r')

# Our guess of the standards. We assume they don't have any parasitics.
oo_i = coax.open(nports=2, name='open')
ss_i = coax.short(nports=2, name='short')
# Match is only measured on one port. Resistance can be different from 50 ohms.
m_i = coax.resistor(R=50, name='r') ** coax.short(nports=1)
# Thru must be known exactly
thru = coax.line(d=100, unit='um', name='thru')

# Actual reflects with parasitics. They must be identical in both ports.
# Short is slightly lossy.
ss = coax.line(d=200, unit='um') ** coax.load(-0.98,nports=2, name='short') ** coax.line(d=200, unit='um')
oo = coax.shunt_capacitor(10e-15) ** coax.open(nports=2, name='open') ** coax.shunt_capacitor(10e-15)

# Match standard has inductance in series.
match_l = 40e-12
l = coax.inductor(L=match_l)
m = l**m_i

# Make two-port of the match with open on the second port.
mm = two_port_reflect(m, coax.open(nports=1))
# Make two-port for the match standard
mm_i = coax.match(nports=2, name='load')

# These are our guesses of the calibration standards.
approx_ideals = [
    thru,
    ss_i,
    oo_i,
    mm_i
    ]

# These are the actual standards with parasitics.
ideals = [
    thru,
    ss,
    oo,
    mm
    ]

# Make measurement of the standards using the random error boxes and switch terms.
measured = [terminate(X**k**Y, gamma_f, gamma_r) for k in ideals]

### 可视化校准标准

In [ ]:
plt.figure()
oo.plot_s_smith(m=0, n=0, label='Open')
ss.plot_s_smith(m=0, n=0, label='Short')
mm.plot_s_smith(m=0, n=0, label='Match')

plt.figure()
oo.plot_s_db(m=0, n=0, label='Open')
ss.plot_s_db(m=0, n=0, label='Short')
mm.plot_s_db(m=0, n=0, label='Match')

### LRRM 校准

我们假装不知道实际的校准标准（包括寄生参数），并且仅提供不包含寄生参数的校准标准近似值以及包含寄生参数的校准标准的测量结果。

In [ ]:
cal = LRRM(
    ideals = approx_ideals,
    measured = measured,
    switch_terms = [gamma_f, gamma_r])

### 可视化已求解的校准标准

LRRM 校准用于求解实际的校准标准。我们可以从 `cal` 对象中获取求解后的校准标准。求解后的校准标准应与上述实际校准标准相匹配。

In [ ]:
plt.figure()
cal.solved_r2.plot_s_smith(m=0, n=0, label='Open')
cal.solved_r1.plot_s_smith(m=0, n=0, label='Short')
cal.solved_m.plot_s_smith(m=0, n=0, label='Match')

plt.figure()
cal.solved_r2.plot_s_db(m=0, n=0, label='Open')
cal.solved_r1.plot_s_db(m=0, n=0, label='Short')
cal.solved_m.plot_s_db(m=0, n=0, label='Match')

匹配电路的计算出的电感值也作为校准输出提供。它以数组的形式给出，但使用默认选项时，会在所有频率上拟合单个电感值。

In [ ]:
solved_match_l = cal.solved_l[0]
print(f'Solved inductance {1e12*solved_match_l:.1f} pH, actual inductance {1e12*match_l:.1f} pH')

### 校准待测器件

可以使用 `apply_cal` 方法对测量的DUT进行校准。S参数应该完全匹配。

In [ ]:
# Let's generate a DUT: 5 mm long 75 ohm line.
dut = coax.line(d=5, unit='mm', z0=75)

dut_measured = terminate(X**dut**Y, gamma_f, gamma_r)
dut_cal = cal.apply_cal(dut_measured)

plt.figure()
dut.plot_s_db(m=0, n=0, label='Actual S11')
dut.plot_s_db(m=1, n=0, label='Actual S21')
dut_cal.plot_s_db(m=0, n=0, label='Calibrated S11')
dut_cal.plot_s_db(m=1, n=0, label='Calibrated S21')
plt.ylim([-20, 5])

### 使用反射 |S11| 进行校准验证在校准过程中，假设第二个反射 |S11| 是已知的（在本例中，|S11| = 1），但在将单个电感拟合到匹配标准时，这个假设可能会被打破。如果实际的匹配不能很好地建模为与电感串联的已知电阻，则会导致反射标准的无损性被破坏。通过绘制反射的绝对值，我们可以了解校准假设的准确性。让我们首先绘制先前校准中的开路 |S11|。如果一切正常，它应该正好是 0 dB。

In [ ]:
plt.figure()
cal.solved_r2.plot_s_db(m=0, n=0, label='Solved open')
plt.ylim([-0.01, 0.01])

### 使用电容匹配进行校准匹配网络的 LRRM 校准模型是一个与电感串联的电阻。如果匹配网络还具有并联电容，则无法正确求解，并且会产生校正后的测量误差。让我们定义一个新的匹配校准标准，并使用它重新进行校准。

In [ ]:
# Match standard with series inductance and parallel capacitance.
match_c = 20e-15
match_l = 20e-12
l = coax.inductor(L=match_l)
c = coax.shunt_capacitor(match_c)
m = c**l**m_i

# Make two-port of the match with open on the second port.
mm = two_port_reflect(m, coax.open(nports=1))

# Redo the match measurement
ideals[3] = mm
measured[3] = terminate(X**mm**Y, gamma_f, gamma_r)

# Redo the calibration
cal = LRRM(
    ideals = approx_ideals,
    measured = measured,
    switch_terms = [gamma_f, gamma_r]
    )

### 可视化电容匹配和求解后的匹配结果

校准试图尽可能地使电感与匹配电路相匹配，但它无法仅使用电感来精确地模拟匹配电路。最接近的匹配是通过一个具有负值的电感来实现的。

In [ ]:
plt.figure()
mm.plot_s_smith(m=0, n=0, label='Actual')
cal.solved_m.plot_s_smith(m=0, n=0, label='Solved')

plt.figure()
mm.plot_s_db(m=0, n=0, label='Actual')
cal.solved_m.plot_s_db(m=0, n=0, label='Solved')

In [ ]:
solved_match_l = cal.solved_l[0]
print(f'Solved inductance {1e12*solved_match_l:.1f} pH')

绘制开路端口的 |S11| 值，结果显示，经过处理后的开路端口并非无损，这表明一些校准假设可能没有得到满足。

In [ ]:
plt.figure()
cal.solved_r2.plot_s_db(m=0, n=0, label='Solved open')

### 使用不正确的匹配模型进行DUT（待测设备）测量

不正确的匹配会导致校准参数出现误差。在高频率下，由于匹配模型误差增大，误差也会随之增加。

In [ ]:
dut_cal2 = cal.apply_cal(dut_measured)

plt.figure()
dut.plot_s_db(m=0, n=0, label='Actual S11')
dut.plot_s_db(m=1, n=0, label='Actual S21')
dut_cal2.plot_s_db(m=0, n=0, label='Calibrated S11')
dut_cal2.plot_s_db(m=1, n=0, label='Calibrated S21')
plt.ylim([-20, 5])

### 使用电感和电容进行匹配拟合LRRM 具有使用并联电容匹配模型的选项，这使得可以拟合上述匹配。此拟合方法的附加要求是，第二次反射应该是开路，并且具有一些未知的电容。首先拟合开路电容，假设匹配是完全电阻性的，并且在低频时，该假设可能更准确。当开路电容已知时，拟合匹配电容和电感。对开路和匹配进行迭代几次，以优化开路和匹配的估计值。可以通过将 `match_fit = 'lc'` 传递给校准方法来使用此拟合方法。

In [ ]:
# Redo the calibration using LC match model
cal = LRRM(
    ideals = approx_ideals,
    measured = measured,
    match_fit = 'lc',
    switch_terms = [gamma_f, gamma_r]
    )

In [ ]:
plt.figure()
mm.plot_s_smith(m=0, n=0, label='Actual')
cal.solved_m.plot_s_smith(m=0, n=0, label='Solved')

plt.figure()
mm.plot_s_db(m=0, n=0, label='Actual')
cal.solved_m.plot_s_db(m=0, n=0, label='Solved')

In [ ]:
solved_match_l = cal.solved_l[0]
solved_match_c = cal.solved_c[0]
print(f'Solved inductance {1e12*solved_match_l:.1f} pH, actual inductance {1e12*match_l:.1f} pH')
print(f'Solved capacitance {1e15*solved_match_c:.1f} fF, actual capacitance {1e15*match_c:.1f} fF')

现在将校准应用于测量数据，应该能得到较为精确的结果。

In [ ]:
dut_cal3 = cal.apply_cal(dut_measured)

plt.figure()
dut.plot_s_db(m=0, n=0, label='Actual S11')
dut.plot_s_db(m=1, n=0, label='Actual S21')
dut_cal3.plot_s_db(m=0, n=0, label='Calibrated S11')
dut_cal3.plot_s_db(m=1, n=0, label='Calibrated S21')
plt.ylim([-20, 5])

## 与 SOLT 校准的比较

传统的两端口 SOLT 校准假定所有校准标准的值都是精确已知的。我们可以比较，在采用相同且近似已知的校准标准进行测量时，它的性能如何。对于 SOLT 校准，还需要测量第二个端口的匹配情况，我们假设它与第一个端口的匹配情况相同。随机生成的误差框使得校准变得尤为困难。

In [ ]:
# SOLT requires match measurement on both ports.
mm = two_port_reflect(m, m)
measured[3] = terminate(X**mm**Y, gamma_f, gamma_r)

# TwelveTerm assumes that thru is last.
cal12 = TwelveTerm(
    ideals = list(reversed(approx_ideals)),
    measured = list(reversed(measured)),
    n_thrus = 1,
    )

dut_cal12 = cal12.apply_cal(dut_measured)

plt.figure()
dut.plot_s_db(m=0, n=0, label='Actual S11')
dut.plot_s_db(m=1, n=0, label='Actual S21')
dut_cal12.plot_s_db(m=0, n=0, label='Calibrated S11')
dut_cal12.plot_s_db(m=1, n=0, label='Calibrated S21')
plt.ylim([-20, 5])